# FarmGemma Training on Kaggle - FREE GPU

**Runtime**: GPU P100 (16GB) - Free weekly hours

## Steps:
1. **Kaggle Settings** → Accelerator → GPU P100
2. **Internet** → On (for downloading models)
3. Run cells below

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece protobuf google-generativeai
!pip install -q torch torchvision torchaudio
!pip install -q kaggle

import os
print("Dependencies installed!")

In [ ]:
# Step 2: Mount Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/farmgemma/models"
DATA_DIR = "/content/drive/MyDrive/farmgemma/data"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Model dir: {MODEL_DIR}")

In [ ]:
# Step 3: Download Agricultural Dataset from HuggingFace
from datasets import load_dataset

# Load PlantVillage dataset (crop disease images)
print("Loading PlantVillage dataset...")
plant_village = load_dataset("AlifMiah/PlantVillage-Dataset", split="train")
print(f"Loaded {len(plant_village)} images")

# Load agricultural Q&A dataset
print("Loading agricultural Q&A...")
agri_qa = load_dataset("knowrohit07/Indian_Agriculture_Crop_Health")
print(f"Loaded {len(agri_qa)} samples")

In [ ]:
# Step 4: Login to HuggingFace for Gemma access
!pip install -q huggingface_hub
from huggingface_hub import login

# Get token from https://huggingface.co/settings/tokens
HF_TOKEN = input("Enter HuggingFace token: ")
login(token=HF_TOKEN)
print("Logged in to HuggingFace!")

In [ ]:
# Step 5: Load Gemma 3 1B (smaller, faster, works on P100)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Gemma 3 1B model...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
print("Model loaded! Ready for fine-tuning")

In [ ]:
# Step 6: Setup LoRA for efficient fine-tuning
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 7: Prepare training data
def format_agricultural_sample(sample):
    prompt = f"""You are FarmGemma, an AI assistant for Indian farmers.

Question: {question}

Answer: {answer}"""
    return {"text": prompt.format(question=sample['question'], answer=sample['answer'])}

train_data = agri_qa['train'].map(format_agricultural_sample)

In [ ]:
# Step 8: Training configuration
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/farmgemma-1b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    optim="paged_adamw_8bit"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=data_collator
)
print("Training configured!")

In [ ]:
# Step 9: TRAIN! (This will take ~2-3 hours on P100)
trainer.train()

In [ ]:
# Step 10: Save model to Google Drive
trainer.save_model(f"{MODEL_DIR}/farmgemma-1b-final")
tokenizer.save_pretrained(f"{MODEL_DIR}/farmgemma-1b-final")
print(f"Model saved to {MODEL_DIR}/farmgemma-1b-final")

In [ ]:
# Step 11: Test the model
test_question = "मेरे चावल की फसल में भूरे धब्बे हैं, क्या समस्या है?"

prompt = f"""You are FarmGemma, an AI assistant for Indian farmers.

Question: {test_question}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Question:", test_question)
print("\nFarmGemma:", response.split("Answer:")[-1].strip())

## Kaggle Free GPU Limits

| Resource | Free Limit |
|----------|------------|
| GPU | P100 16GB, ~30-40 hrs/week |
| Storage | 20GB temporary |
| Internet | ✅ Available |

**Pro Tips:**
- Save checkpoints to Google Drive frequently
- Use Gemma 3 1B instead of 4B for faster training
- Enable `persist` in gradient checkpointing